In [1]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

import string
from nltk.corpus import stopwords
import re
from nltk.stem import WordNetLemmatizer

from sklearn.feature_extraction.text import CountVectorizer
from sklearn.feature_extraction.text import TfidfVectorizer

 1. Load raw data

In [2]:
data = pd.read_csv(r'C:\Users\jenim\Desktop\DSML\Week7\project-3-nlp\dataset\training_data.csv', sep="\t", header=None, names=["label", "text"])
data.head()

,label,text
0,0,donald trump sends out embarrassing new year‚s...
1,0,drunk bragging trump staffer started russian c...
2,0,sheriff david clarke becomes an internet joke ...
3,0,trump is so obsessed he even has obama‚s name ...
4,0,pope francis just called out donald trump duri...


In [3]:
df = data.copy() # creating a copy to keep a untempered dataset

In [4]:
data.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 34152 entries, 0 to 34151
Data columns (total 2 columns):
 #   Column  Non-Null Count  Dtype 
---  ------  --------------  ----- 
 0   label   34152 non-null  int64 
 1   text    34152 non-null  object
dtypes: int64(1), object(1)
memory usage: 533.8+ KB


In [5]:
data.label.value_counts()

label
0    17572
1    16580
Name: count, dtype: int64

In [6]:
data.duplicated().sum()

np.int64(1946)

In [8]:
data=data.drop_duplicates()

2. Data cleaning

2.1 Diagnostics

In [9]:
def diagnose_text_data(texts):
    issues = {
        'urls': 0,
        'html_tags': 0,
        'special_chars': 0,
        'extra_whitespace': 0,
        'very_short': 0,
        'very_long': 0,
        'numbers_only': 0
    }

    for text in texts[:1000]:  # sample check
        if pd.isna(text):
            continue

        if re.search(r'http\S+|www\S+', text):
            issues['urls'] += 1
        if re.search(r'<[^>]+>', text):
            issues['html_tags'] += 1
        if re.search(r'[^\w\s]', text):  # non-word, non-space chars
            issues['special_chars'] += 1
        if '  ' in text or '\n' in text or '\t' in text:
            issues['extra_whitespace'] += 1
        if len(text.split()) < 3:
            issues['very_short'] += 1
        if len(text.split()) > 500:
            issues['very_long'] += 1
        if text.isdigit():
            issues['numbers_only'] += 1
    
    return issues

In [10]:
problems = diagnose_text_data(data['text'])
print("Text Issues Found:")
for issue, count in problems.items():
    print(f"  {issue}: {count}")

for label in data['label'].unique():
    label_text = data[data['label'] == label]['text']
    problems_label = diagnose_text_data(label_text)
    print(f"\n=== LABEL {label} ===")
    for issue, count in problems_label.items():
        print(f"  {issue}: {count}")

Text Issues Found:
  urls: 0
  html_tags: 0
  special_chars: 655
  extra_whitespace: 0
  very_short: 9
  very_long: 0
  numbers_only: 0

=== LABEL 0 ===
  urls: 0
  html_tags: 0
  special_chars: 655
  extra_whitespace: 0
  very_short: 9
  very_long: 0
  numbers_only: 0

=== LABEL 1 ===
  urls: 0
  html_tags: 0
  special_chars: 704
  extra_whitespace: 190
  very_short: 0
  very_long: 0
  numbers_only: 0


Good data quality, so minimal cleaning is needed. 
Observations:
1. 9 very short texts are found only in the fake news. Consistent with sensationalist headlines.
2. 192 instances od extra whitespace found in real news only.
3. Special characters are consistent.

In [11]:
def clean_text(text):
    if pd.isna(text):
        return ""

    # Remove special characters (keep only letters, numbers, and spaces)
    text = re.sub(r'[^a-zA-Z0-9\s]', '', text)
    
    # Normalize extra whitespace (multiple spaces, tabs, newlines → single space)
    text = re.sub(r'\s+', ' ', text)
    
    # Remove leading/trailing whitespace
    text = text.strip()
    
    # Lowercase
    text = text.lower()
    
    return text

In [12]:
data['text'] = data['text'].apply(clean_text)
print(f"Dataset shape: {data.shape}")
print(data['text'].head(10))

Dataset shape: (32206, 2)
0    donald trump sends out embarrassing new years ...
1    drunk bragging trump staffer started russian c...
2    sheriff david clarke becomes an internet joke ...
3    trump is so obsessed he even has obamas name c...
4    pope francis just called out donald trump duri...
5    racist alabama cops brutalize black boy while ...
6                            fresh off the golf course
7    trump said some insanely racist stuff inside t...
8     former cia director slams trump over un bullying
9    brandnew protrump ad features so much a kissin...
Name: text, dtype: object


3. Data splitting

In [13]:
X = data['text']
y = data['label']

X_train, X_test, y_train, y_test = train_test_split(X,y, test_size=0.2, random_state=42, stratify=data['label'])

print('X_train shape:', X_train.shape)
print('X_test shape:', X_test.shape)

X_train shape: (25764,)
X_test shape: (6442,)


4. Tokenization for data Sets

In [14]:
import nltk
from nltk.tokenize import word_tokenize

nltk.download('punkt')

[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\jenim\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!


True

In [15]:
X_train_token = X_train.apply(word_tokenize)
print(X_train_token.iloc[5])

X_test_token = X_test.apply(word_tokenize)
print(X_test_token.iloc[5])

['pop', 'star', 'ariana', 'grande', 'says', 'i', 'hate', 'americans', 'i', 'hate', 'america', 'words', 'were', 'taken', 'out', 'of', 'context', 'video']
['trump', 'calls', 'global', 'currency', 'devaluations', 'dangerous', 'for', 'us']


5. Stopwords Removal

In [16]:
stop_words = set(stopwords.words('english'))

X_train_no_stopword = X_train_token.apply(lambda tokens:[word for word in tokens if word.lower() not in stop_words])
X_test_no_stopword = X_test_token.apply(lambda tokens:[word for word in tokens if word.lower() not in stop_words])


In [17]:
print("=== COMPARISON ===")
print("\nBefore stopword removal (first 15 words):")
print(X_train_token.iloc[0][:15])

print("\nAfter stopword removal (first 15 words):")
print(X_train_no_stopword.iloc[0][:15])

print(f"\nTraining set shape: {X_train_no_stopword.shape}")
print(f"Test set shape: {X_test_no_stopword.shape}")

=== COMPARISON ===

Before stopword removal (first 15 words):
['clinton', 'thinks', 'regulators', 'should', 'scrutinize', 'atttime', 'warner', 'deal', 'spokesman']

After stopword removal (first 15 words):
['clinton', 'thinks', 'regulators', 'scrutinize', 'atttime', 'warner', 'deal', 'spokesman']

Training set shape: (25764,)
Test set shape: (6442,)


6. Lemmatization

In [18]:
word_lemmatizer = WordNetLemmatizer()

X_train_lemmatized = X_train_no_stopword.apply(lambda tokens: [word_lemmatizer.lemmatize(word) for word in tokens])
X_test_lemmatized = X_test_no_stopword.apply(lambda tokens: [word_lemmatizer.lemmatize(word) for word in tokens])


In [19]:
print("=== BEFORE LEMMATIZATION ===")
print("Training example (first 15 words):")
print(X_train_no_stopword.iloc[0][:15])

print("\n=== AFTER LEMMATIZATION ===")
print("Training example (first 15 words):")
print(X_train_lemmatized.iloc[0][:15])

print(f"\nTraining data shape: {X_train_lemmatized.shape}")
print(f"Test data shape: {X_test_lemmatized.shape}")

=== BEFORE LEMMATIZATION ===
Training example (first 15 words):
['clinton', 'thinks', 'regulators', 'scrutinize', 'atttime', 'warner', 'deal', 'spokesman']

=== AFTER LEMMATIZATION ===
Training example (first 15 words):
['clinton', 'think', 'regulator', 'scrutinize', 'atttime', 'warner', 'deal', 'spokesman']

Training data shape: (25764,)
Test data shape: (6442,)


7. Word Frequency

In [20]:
from collections import Counter

In [21]:
fake_train_docs = X_train_lemmatized[y_train == 0]
real_train_docs = X_train_lemmatized[y_train == 1]

# Flatten tokens
fake_all_tokens = [token for doc in fake_train_docs for token in doc]
real_all_tokens = [token for doc in real_train_docs for token in doc]

fake_word_counts = Counter(fake_all_tokens)
real_word_counts = Counter(real_all_tokens)

top_10_fake = fake_word_counts.most_common(10)
top_10_real = real_word_counts.most_common(10)


In [22]:
print("Top 10 words in Fake news:")
for i, (word, count) in enumerate(top_10_fake, 1):
    print(f"  {i}. '{word}': {count} ")

print("\nTop 10 words in Real news:")
for i, (word, count) in enumerate(top_10_real, 1):
    print(f"  {i}. '{word}': {count} ")

Top 10 words in Fake news:
  1. 'trump': 5399 
  2. 'video': 4023 
  3. 'hillary': 1056 
  4. 'obama': 905 
  5. 'republican': 615 
  6. 'clinton': 581 
  7. 'get': 543 
  8. 'gop': 541 
  9. 'president': 540 
  10. 'tweet': 536 

Top 10 words in Real news:
  1. 'trump': 3975 
  2. 'u': 2749 
  3. 'say': 1862 
  4. 'house': 1123 
  5. 'republican': 764 
  6. 'white': 621 
  7. 'senate': 580 
  8. 'russia': 570 
  9. 'bill': 536 
  10. 'tax': 523 


In [23]:
X_train_processed = X_train_lemmatized.apply(lambda tokens: ' '.join(tokens))

X_test_processed = X_test_lemmatized.apply(lambda tokens: ' '.join(tokens))

print("Sample processed text:")
print(X_train_processed.iloc[5])


Sample processed text:
pop star ariana grande say hate american hate america word taken context video


8. Bag of Words (Countvectorizer)

In [24]:
vectorizer= CountVectorizer()

X_train_bow = vectorizer.fit_transform(X_train_processed)

X_test_bow = vectorizer.transform(X_test_processed)



9. TF-IDF

In [25]:
tfidf_vectorizer = TfidfVectorizer()

X_train_tfidf = tfidf_vectorizer.fit_transform(X_train_processed)
X_test_tfidf = tfidf_vectorizer.transform(X_test_processed)


In [26]:
sample_vector = X_train_tfidf[0].toarray()[0]

feature_names = tfidf_vectorizer.get_feature_names_out()

print(f"Original text:\n{X_train_processed.iloc[0]}\n")

# Words with TF-IDF weights (only non-zero)
print("Words with TF-IDF weights (non-zero):")
non_zero_indices = sample_vector.nonzero()[0]
for idx in non_zero_indices:
    print(f"  {feature_names[idx]:20} : {sample_vector[idx]:.4f}")

Original text:
clinton think regulator scrutinize atttime warner deal spokesman

Words with TF-IDF weights (non-zero):
  atttime              : 0.4471
  clinton              : 0.1912
  deal                 : 0.2361
  regulator            : 0.3580
  scrutinize           : 0.4788
  spokesman            : 0.3123
  think                : 0.2834
  warner               : 0.4153


**Step-10. Classifier training**

## 1. **Model-1: Logistic Regression with default hyperparameters in TF-IDF and classifier**

In [27]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix, ConfusionMatrixDisplay


In [28]:
model1 = LogisticRegression()
model1.fit(X_train_tfidf, y_train)

,penalty,'l2'
,dual,False
,tol,0.0001
,C,1.0
,fit_intercept,True
,intercept_scaling,1
,class_weight,None
,random_state,None
,solver,'lbfgs'
,max_iter,100
,multi_class,'deprecated'


In [29]:
y_pred_train = model1.predict(X_train_tfidf)
y_pred_test = model1.predict(X_test_tfidf)

accuracy1_test = accuracy_score(y_test, y_pred_test)
print(f"Test Accuracy: {accuracy1_test:.4f}\n")
accuracy1_train = accuracy_score(y_train, y_pred_train)
print(f"Train Accuracy: {accuracy1_train:.4f}\n")

cm1 = confusion_matrix(y_test, y_pred_test)
print(cm1)

report1 = classification_report(y_test, y_pred_test)
print(report1)

Test Accuracy: 0.9252

Train Accuracy: 0.9544

[[2908  297]
 [ 185 3052]]
              precision    recall  f1-score   support

           0       0.94      0.91      0.92      3205
           1       0.91      0.94      0.93      3237

    accuracy                           0.93      6442
   macro avg       0.93      0.93      0.93      6442
weighted avg       0.93      0.93      0.93      6442



In [30]:
import pickle

# Save the model
with open('logistic_regression_model.pkl', 'wb') as f:
    pickle.dump(model1, f)

# Save the vectorizer too (needed for new predictions)
with open('tfidf_vectorizer.pkl', 'wb') as f:
    pickle.dump(vectorizer, f)

**Grid Search- for best estimators of hyperparameters**

In [31]:
from sklearn.model_selection import GridSearchCV
param_grid = {
    'C': [0.1, 1, 3, 10, 100],
    'penalty': ['l1', 'l2'],
    'solver': ['liblinear']
}
grid_search = GridSearchCV(
    LogisticRegression(max_iter=2000, random_state=42),
    param_grid,
    cv=5,
    scoring='f1',
    n_jobs=-1
)
print("Testing all combinations...")
grid_search.fit(X_train_tfidf, y_train)


best_model = grid_search.best_estimator_

y_pred_test = best_model.predict(X_test_tfidf)
test_acc = accuracy_score(y_test, y_pred_test)

y_pred_train = best_model.predict(X_train_tfidf)
train_acc = accuracy_score(y_train, y_pred_train)

print(f"Best parameters: {grid_search.best_params_}")
print(f"Test Accuracy: {test_acc:.4f}({test_acc*100:.2f}%)")
print(f"Train Accuracy: {train_acc:.4f}({train_acc*100:.2f}%)")
print(f"Overfitting: {((train_acc - test_acc)*100):.2f}%")

print(classification_report(y_test, y_pred_test))

Testing all combinations...
Best parameters: {'C': 3, 'penalty': 'l2', 'solver': 'liblinear'}
Test Accuracy: 0.9301(93.01%)
Train Accuracy: 0.9714(97.14%)
Overfitting: 4.12%
              precision    recall  f1-score   support

           0       0.94      0.92      0.93      3205
           1       0.92      0.94      0.93      3237

    accuracy                           0.93      6442
   macro avg       0.93      0.93      0.93      6442
weighted avg       0.93      0.93      0.93      6442



# Extra: Testing with CountVectorizer in Logistic Regression Model

In [56]:
model2_2 = LogisticRegression(C=1, max_iter=1000, random_state=42)
model2_2.fit(X_train_bow, y_train)

,penalty,'l2'
,dual,False
,tol,0.0001
,C,1
,fit_intercept,True
,intercept_scaling,1
,class_weight,None
,random_state,42
,solver,'lbfgs'
,max_iter,1000
,multi_class,'deprecated'


In [57]:
y_pred_train2 = model2_2.predict(X_train_bow)
y_pred_test2 = model2_2.predict(X_test_bow)

accuracy_test2 = accuracy_score(y_test, y_pred_test2)
print(f"Test Accuracy: {accuracy_test2:.4f}\n")
accuracy_train2 = accuracy_score(y_train, y_pred_train2)
print(f"Train Accuracy: {accuracy_train2:.4f}\n")

cm2 = confusion_matrix(y_test, y_pred_test2)
print(cm1)

report1 = classification_report(y_test, y_pred_test2)
print(report1)

Test Accuracy: 0.9295

Train Accuracy: 0.9752

[[2908  297]
 [ 185 3052]]
              precision    recall  f1-score   support

           0       0.94      0.91      0.93      3205
           1       0.92      0.95      0.93      3237

    accuracy                           0.93      6442
   macro avg       0.93      0.93      0.93      6442
weighted avg       0.93      0.93      0.93      6442



## 2. **Model-2: Logistic regression model with regularization C= 3 and TF-IDF max_df = 0.8**

In [32]:
vectorizer_new = TfidfVectorizer(
    max_features=5000,
    max_df=0.8,         # Remove words in > 80% of documents
    min_df=2,            # Remove words in < 2 documents
    ngram_range=(1,2)
)
X_train_tfidf_new = vectorizer_new.fit_transform(X_train_processed)
X_test_tfidf_new = vectorizer_new.transform(X_test_processed)

model_new = LogisticRegression(C=3, max_iter=1000, random_state=42)
model_new.fit(X_train_tfidf_new, y_train)

y_pred_train_new = model_new.predict(X_train_tfidf_new)
y_pred_test_new = model_new.predict(X_test_tfidf_new)

train_acc_new = accuracy_score(y_train, y_pred_train_new)
test_acc_new = accuracy_score(y_test, y_pred_test_new)

print("NEW MODEL PERFORMANCE (WITH max_df)")
print("="*60)
print(f"Test Accuracy: {test_acc_new:.4f} ({test_acc_new*100:.2f}%)")
print(f"Train Accuracy: {train_acc_new:.4f} ({train_acc_new*100:.2f}%)")
print(f"Difference: {((train_acc_new - test_acc_new)*100):.2f}%")

print(classification_report(y_test, y_pred_test_new))

NEW MODEL PERFORMANCE (WITH max_df)
Test Accuracy: 0.9263 (92.63%)
Train Accuracy: 0.9596 (95.96%)
Difference: 3.33%
              precision    recall  f1-score   support

           0       0.94      0.91      0.92      3205
           1       0.91      0.94      0.93      3237

    accuracy                           0.93      6442
   macro avg       0.93      0.93      0.93      6442
weighted avg       0.93      0.93      0.93      6442



In [33]:
# Save the model
with open('logistic_regression_model2_max_df.pkl', 'wb') as f:
    pickle.dump(model_new, f)

# Save the vectorizer too (needed for new predictions)
with open('tfidf_vectorizer_new.pkl', 'wb') as f:
    pickle.dump(vectorizer_new, f)

## 3. **Model-3: Multinomial Naive Bayes**


In [34]:
from sklearn.naive_bayes import MultinomialNB

In [35]:
vectorizer_mnb = TfidfVectorizer(
    max_features=5000,
    max_df=0.8,         # Remove words in > 80% of documents
    min_df=2,            # Remove words in < 2 documents
    ngram_range=(1,2)
)
X_train_vec_mnb = vectorizer_mnb.fit_transform(X_train_processed)
X_test_vec_mnb = vectorizer_mnb.transform(X_test_processed)

model_mnb = MultinomialNB()
model_mnb.fit(X_train_vec_mnb, y_train)


,alpha,1.0
,force_alpha,True
,fit_prior,True
,class_prior,None


In [36]:
y_pred_train_mnb = model_mnb.predict(X_train_vec_mnb)
y_pred_test_mnb = model_mnb.predict(X_test_vec_mnb)

train_acc_mnb = accuracy_score(y_train, y_pred_train_mnb)
test_acc_mnb = accuracy_score(y_test, y_pred_test_mnb)

print("MNB MODEL PERFORMANCE (WITH max_df)")
print("="*60)
print(f"Test Accuracy: {test_acc_mnb:.4f} ({test_acc_mnb*100:.2f}%)")
print(f"Train Accuracy: {train_acc_mnb:.4f} ({train_acc_mnb*100:.2f}%)")
print(f"Difference: {(train_acc_mnb - test_acc_mnb)*100:.2f}%")

print(classification_report(y_test, y_pred_test_mnb))

MNB MODEL PERFORMANCE (WITH max_df)
Test Accuracy: 0.9193 (91.93%)
Train Accuracy: 0.9344 (93.44%)
Difference: 1.52%
              precision    recall  f1-score   support

           0       0.92      0.92      0.92      3205
           1       0.92      0.92      0.92      3237

    accuracy                           0.92      6442
   macro avg       0.92      0.92      0.92      6442
weighted avg       0.92      0.92      0.92      6442



## 4. **Model-4: Light GBM with Word2Vec embedding**

In [35]:
pip install gensim

Note: you may need to restart the kernel to use updated packages.


In [36]:
pip install lightgbm

Note: you may need to restart the kernel to use updated packages.


In [37]:
from gensim.models import Word2Vec
from lightgbm import LGBMClassifier

In [38]:
w2v_model = Word2Vec(
    sentences=X_train_lemmatized,
    vector_size=100,      # Each word = 100 numbers
    window=5,             # Context window
    workers=4,
    epochs=5
)

In [39]:
w2v_model.save('word2vec_model.model')

In [40]:
import numpy as np

In [41]:
def text_to_w2v_vector(tokens, model, vector_size=100):
    vectors = []
    for token in tokens:
        if token in model.wv:
            vectors.append(model.wv[token])
    
    if len(vectors) == 0:
        return np.zeros(vector_size)
    
    # Return average of all word vectors
    return np.mean(vectors, axis=0)

X_train_w2v = np.array([text_to_w2v_vector(tokens, w2v_model) for tokens in X_train_lemmatized])
X_test_w2v = np.array([text_to_w2v_vector(tokens, w2v_model)for tokens in X_test_lemmatized])


In [42]:
lgb_model = LGBMClassifier(
    n_estimators=100,        # Number of boosting stages
    max_depth=6,             # Maximum tree depth
    learning_rate=0.1,       # Learning rate (shrinkage)
    random_state=42,
    n_jobs=-1,
    verbose=-1
)
lgb_model.fit(X_train_w2v, y_train)


,boosting_type,'gbdt'
,num_leaves,31
,max_depth,6
,learning_rate,0.1
,n_estimators,100
,subsample_for_bin,200000
,objective,None
,class_weight,None
,min_split_gain,0.0
,min_child_weight,0.001
,min_child_samples,20


In [43]:
y_pred_train = lgb_model.predict(X_train_w2v)
y_pred_test = lgb_model.predict(X_test_w2v)
train_acc = accuracy_score(y_train, y_pred_train)
test_acc = accuracy_score(y_test, y_pred_test)
overfitting = train_acc - test_acc

print(f"\nTrain Accuracy: {train_acc:.4f} ({train_acc*100:.2f}%)")
print(f"Test Accuracy:  {test_acc:.4f} ({test_acc*100:.2f}%)")
print(f"Difference: {overfitting*100:.2f}%")

cm = confusion_matrix(y_test, y_pred_test)
print(cm)
print(classification_report(y_test, y_pred_test, target_names=['Fake (0)', 'Real (1)']))



Train Accuracy: 0.9111 (91.11%)
Test Accuracy:  0.8722 (87.22%)
Difference: 3.88%
[[2771  434]
 [ 389 2848]]
              precision    recall  f1-score   support

    Fake (0)       0.88      0.86      0.87      3205
    Real (1)       0.87      0.88      0.87      3237

    accuracy                           0.87      6442
   macro avg       0.87      0.87      0.87      6442
weighted avg       0.87      0.87      0.87      6442



**Test data: Testing Models on the testing data**


In [58]:
test_data = pd.read_csv(r'dataset/testing_data.csv', sep='\t', header=None, names=['label', 'text'])
test_data.head()


,label,text
0,2,copycat muslim terrorist arrested with assault...
1,2,wow! chicago protester caught on camera admits...
2,2,germany's fdp look to fill schaeuble's big shoes
3,2,mi school sends welcome back packet warning ki...
4,2,u.n. seeks 'massive' aid boost amid rohingya '...


In [59]:
def clean_text(text):
    if pd.isna(text):
        return ""
    
    # Normalize extra whitespace
    text = re.sub(r'\s+', ' ', text)
    text = text.strip()
    
    # Lowercase
    text = text.lower()
    
    # Remove special characters (keep only letters, numbers, spaces)
    text = re.sub(r'[^a-zA-Z0-9\s]', '', text)
    
    return text

test_data['text'] = test_data['text'].apply(clean_text)


In [60]:
test_data_tokenized = test_data['text'].apply(word_tokenize)
test_data_tokenized

0       [copycat, muslim, terrorist, arrested, with, a...
1       [wow, chicago, protester, caught, on, camera, ...
2       [germanys, fdp, look, to, fill, schaeubles, bi...
3       [mi, school, sends, welcome, back, packet, war...
4       [un, seeks, massive, aid, boost, amid, rohingy...
                              ...                        
9979    [boom, fox, news, leftist, chris, wallace, att...
9980    [here, it, is, list, of, democrat, hypocrites,...
9981    [new, fires, ravage, rohingya, villages, in, n...
9982    [meals, on, wheels, shuts, the, lyin, lefties,...
9983    [brilliant, tucker, carlson, and, ayaan, hirsi...
Name: text, Length: 9984, dtype: object

In [61]:
stop_words = set(stopwords.words('english'))
test_data_no_stopwords = test_data_tokenized.apply(
    lambda tokens: [word for word in tokens if word.lower() not in stop_words]
)
print(test_data_no_stopwords.iloc[2][:15])


['germanys', 'fdp', 'look', 'fill', 'schaeubles', 'big', 'shoes']


In [62]:
lemmatizer = WordNetLemmatizer()

test_data_lemmatized = test_data_no_stopwords.apply(
    lambda tokens: [lemmatizer.lemmatize(word) for word in tokens]
)
print(test_data_lemmatized.iloc[2][:15])

['germany', 'fdp', 'look', 'fill', 'schaeubles', 'big', 'shoe']


In [63]:
test_data_processed = test_data_lemmatized.apply(lambda tokens: ' '.join(tokens))
print(test_data_processed.iloc[2])


germany fdp look fill schaeubles big shoe


# **Test data with Model-2: The best performing Model is the Logistic Regression model with fine tuned TF-IDF hyperparameters**

In [54]:
with open('tfidf_vectorizer_new.pkl', 'rb') as f:
    vectorizer = pickle.load(f)

In [55]:
test_data_tfidf2 = vectorizer.transform(test_data_processed)


In [56]:
with open('logistic_regression_model2_max_df.pkl', 'rb') as f:
    model2 = pickle.load(f)

In [57]:
y_pred2 = model2.predict(test_data_tfidf2)
y_pred2_proba = model2.predict_proba(test_data_tfidf2)


# Create results dataframe
results2 = pd.DataFrame({
    'Original_Text': test_data['text'].values,
    'Predicted_Label': y_pred2,
    'Predicted_Class': ['Fake' if pred == 0 else 'Real' for pred in y_pred2],
    'Confidence_Fake': y_pred2_proba[:, 0],
    'Confidence_Real': y_pred2_proba[:, 1],
    'Confidence_Score': y_pred2_proba.max(axis=1)
})
results2.head(10)

,Original_Text,Predicted_Label,Predicted_Class,Confidence_Fake,Confidence_Real,Confidence_Score
0,copycat muslim terrorist arrested with assault...,0,Fake,0.975359,0.024641,0.975359
1,wow chicago protester caught on camera admits ...,0,Fake,0.977770,0.022230,0.977770
2,germanys fdp look to fill schaeubles big shoes,1,Real,0.418463,0.581537,0.581537
3,mi school sends welcome back packet warning ki...,0,Fake,0.978109,0.021891,0.978109
4,un seeks massive aid boost amid rohingya emerg...,1,Real,0.003968,0.996032,0.996032
5,did oprah just leave nasty hillary wishing she...,0,Fake,0.996695,0.003305,0.996695
6,frances macron says his job not cool cites tal...,1,Real,0.001095,0.998905,0.998905
7,flashback chilling 60 minutes interview with g...,0,Fake,0.969330,0.030670,0.969330
8,spanish foreign ministry says to expel north k...,1,Real,0.011390,0.988610,0.988610
9,trump says cuba did some bad things aimed at u...,1,Real,0.049931,0.950069,0.950069


## 5. Extra **Transformers**


In [58]:
pip install datasets

Note: you may need to restart the kernel to use updated packages.


In [59]:
df.duplicated().sum()

np.int64(1946)

In [60]:
df.head()

,label,text
0,0,donald trump sends out embarrassing new year‚s...
1,0,drunk bragging trump staffer started russian c...
2,0,sheriff david clarke becomes an internet joke ...
3,0,trump is so obsessed he even has obama‚s name ...
4,0,pope francis just called out donald trump duri...


In [61]:
from transformers import pipeline
import torch
from transformers import AutoTokenizer, AutoModelForSequenceClassification, TrainingArguments, Trainer
from datasets import Dataset


In [62]:
classifier = pipeline("text-classification", model="mrm8488/distilroberta-finetuned-financial-news-sentiment-analysis")

Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

In [63]:
predictions = []
for text, label in zip(df['text'], df['label']):
    result = classifier(text)
    predictions.append({
        'predicted': result[0]['label'],
        'true': label
    })

results_df = pd.DataFrame(predictions)


KeyboardInterrupt: 

In [ ]:
y_true = results_df['true'].values
y_pred = results_df['predicted'].map(lambda x: 0 if x == 'NEGATIVE' else 1).values

accuracy = accuracy_score(y_true, y_pred)

print(f"Accuracy: {accuracy:.4f}")
print(classification_report(y_true, y_pred))

Accuracy: 0.4855
              precision    recall  f1-score   support

           0       0.00      0.00      0.00     17572
           1       0.49      1.00      0.65     16580

    accuracy                           0.49     34152
   macro avg       0.24      0.50      0.33     34152
weighted avg       0.24      0.49      0.32     34152



c:\Users\jenim\anaconda3\Lib\site-packages\sklearn\metrics\_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
c:\Users\jenim\anaconda3\Lib\site-packages\sklearn\metrics\_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
c:\Users\jenim\anaconda3\Lib\site-packages\sklearn\metrics\_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
